# Reasoning without observation (ReWOO)

Traditional agent architectures follow a reasoning-acting-observing loop: the agent thinks about what to do, executes an action, observes the result, then thinks about the next action. While this reactive approach is intuitive, it has significant drawbacks. Each action requires waiting for results before planning the next step, leading to sequential dependencies that slow down execution. The constant back-and-forth between reasoning and acting creates high token usage as context must be maintained across multiple LLM calls. The agent cannot parallelize independent actions since it doesn't plan ahead.

Reasoning without Observation (ReWOO) inverts this paradigm by separating planning from execution. The agent first creates a complete action plan with all necessary tool calls identified upfront, then executes all actions (potentially in parallel), and finally synthesizes the results into an answer. This decoupling brings substantial benefits: independent tools can be called in parallel, the number of LLM calls is reduced, and the planning phase can optimize the action sequence before any execution begins.

In this notebook, we will implement the ReWOO framework that demonstrates how planning-before-execution improves efficiency and enables parallelization. We will build a system that takes a complex query, generates a complete action plan identifying all necessary information gathering steps, executes those steps efficiently, and synthesizes the collected information into a comprehensive answer. This approach is particularly valuable for multi-step research tasks, complex calculations requiring multiple tool calls, and any scenario where planning overhead is amortized across many actions.

In [1]:
import os
import re
import math
from typing import List, Dict
from collections import namedtuple

import wikipedia
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

### Initialize the language model

In [2]:
# Temperature 0 gives deterministic plans — the same query produces the same plan every time
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)

Temperature 0 is appropriate for the planning role: we want the LLM to commit to a definite, reproducible plan rather than explore alternatives. The same instance also handles the final synthesis call in phase 3, where determinism produces consistent answers.

## Define the tools
ReWOO's tools are plain Python functions - there is no need for a special wrapper class or framework adapter. The planner refers to tools by name in the plan it generates; the executor looks them up in a registry dict and calls them directly with the resolved input string. Three tools cover the common needs of a research agent: factual lookup, arithmetic, and open-ended reasoning.

In [3]:
def wikipedia_search(query: str) -> str:
    """Return a 3-sentence Wikipedia summary for the given query."""
    try:
        return wikipedia.summary(query, sentences=3)
    except wikipedia.exceptions.DisambiguationError as e:
        # Query matched multiple pages — retry with the first suggestion
        try:
            return wikipedia.summary(e.options[0], sentences=3)
        except Exception:
            return f"Ambiguous query. Options: {', '.join(e.options[:3])}"
    except Exception as e:
        return f"Wikipedia error: {e}"


def calculator(expression: str) -> str:
    """Evaluate a mathematical expression and return the result as a string."""
    try:
        # Restrict builtins to prevent code injection; expose only math and safe callables
        result = eval(
            expression,
            {"__builtins__": {}},
            {"math": math, "abs": abs, "round": round}
        )
        return str(result)
    except Exception as e:
        return f"Calculator error: {e}"


def llm_query(question: str) -> str:
    """Ask the LLM for reasoning, analysis, or extraction from previous results."""
    response = llm.invoke([HumanMessage(content=question)])
    return response.content


# Registry that maps plan-time tool names to their callable functions.
# The planner uses these short names when it writes steps like #E1 = WikiSearch[...].
TOOLS: Dict = {
    "WikiSearch": wikipedia_search,
    "Calculator": calculator,
    "LLMQuery":   llm_query,
}

- **`wikipedia_search`** catches `DisambiguationError` separately from the general case, because disambiguation is a predictable failure mode when the LLM generates a broad query. Retrying with the first suggestion is usually the right heuristic.
- **`calculator`** uses `eval` with a stripped `__builtins__` dict and an explicit allowlist of names. This prevents arbitrary code execution while supporting all standard mathematical expressions and the full `math` module.
- **`llm_query`** is the escape hatch for anything that is not a lookup or arithmetic: extracting a number from a paragraph of Wikipedia text, rephrasing a question, or doing qualitative analysis. It uses the same `llm` instance as the planner.
- **`TOOLS`** is a plain `dict`. The executor looks tools up by name at runtime, so adding a new tool requires only one line.

In [4]:
# Verify each tool works correctly before using them in plans
print("WikiSearch:")
print(wikipedia_search("Eiffel Tower"))
print()

print("Calculator:")
print(calculator("math.sqrt(144) + 100 / 4"))
print()

print("LLMQuery:")
print(llm_query("In one sentence, what is the capital of France?"))

WikiSearch:
The Eiffel Tower (  EYE-fəl; French: Tour Eiffel [tuʁ ɛfɛl] ) is a lattice tower on the Champ de Mars in Paris, France. It is named after the engineer Gustave Eiffel, whose company designed and built the tower from 1887 to 1889.
Locally nicknamed "La dame de fer" (French for "Iron Lady") for its use of wrought iron, it was constructed as the centrepiece of the 1889 World's Fair, and to crown the centennial anniversary of the French Revolution.

Calculator:
37.0

LLMQuery:
The capital of France is Paris.


## Represent a plan step and parse the LLM's output
The planner LLM produces text that interleaves human-readable annotations with structured step lines:
```
Plan: Look up the population of France
#E1 = WikiSearch[France population]
Plan: Extract the numeric value from the Wikipedia text
#E2 = LLMQuery[From this text: #E1 — what is France's population as a plain integer?]
Plan: Divide by 13
#E3 = Calculator[#E2 / 13]
```

Each `#En = ToolName[tool input]` line is one execution step. The `Plan:` annotation lines are for human readability and are ignored by the parser. Three things matter for execution: the variable name (`#E1`), the tool to call (`WikiSearch`), and the raw input string (`France population`). The input string may contain earlier variable names as placeholders — `#E2` above contains `#E1` — but those references are left as-is during parsing. They are resolved later, step by step, during execution.

Using square brackets `[...]` as delimiters for the tool input is a deliberate choice: unlike parentheses with key-value pairs, square brackets are unambiguous when the input string contains commas, quotes, or embedded variable references. A single regex captures the whole input in one group.

In [5]:
# A plan step is fully described by three things: the variable that will hold the result,
# the tool to call, and the raw input string (which may contain #E references).
PlanStep = namedtuple('PlanStep', ['variable', 'tool', 'input'])


def parse_plan(plan_text: str) -> List[PlanStep]:
    """
    Extract PlanStep objects from the raw text the planner LLM produces.

    Only lines matching '#En = ToolName[...]' are parsed.
    'Plan:' annotation lines are silently ignored.

    Args:
        plan_text: Raw text output from the planner LLM call.

    Returns:
        Ordered list of PlanStep namedtuples.
    """
    steps = []
    # Match lines of the form:  #E1 = WikiSearch[France population]
    # Group 1 → variable name (#E1, #E2, ...)
    # Group 2 → tool name    (WikiSearch, Calculator, LLMQuery)
    # Group 3 → raw input    (may include #E references to earlier steps)
    pattern = re.compile(r'(#E\d+)\s*=\s*(\w+)\[([^\]]*)\]')
    for m in pattern.finditer(plan_text):
        steps.append(PlanStep(
            variable=m.group(1),
            tool=m.group(2),
            input=m.group(3).strip(),
        ))
    return steps

- **`PlanStep`** is a `namedtuple` with exactly three fields - `variable`, `tool`, `input` - and nothing else. The original implementation used a full `@dataclass` with six fields including a `dependencies` list. That list is derived information (which `#E` names appear in `input`?) that is better computed at substitution time rather than stored redundantly at parse time.
- **`parse_plan`** uses a single regex. The pattern `(#E\d+)\s*=\s*(\w+)\[([^\]]*)\]` relies on square brackets as delimiters: `[^\]]*` matches anything except `]`, so the entire tool input - commas, spaces, embedded `#E` references - is captured in group 3 without any special handling. Variable references inside the input are left untouched; resolving them is the executor's job.

In [6]:
# Test the parser with a manually written plan that includes a variable reference
sample_plan = """
Plan: Look up France's population
#E1 = WikiSearch[France population]
Plan: Extract just the number from the text
#E2 = LLMQuery[From this text: #E1 — what is France's population as a plain integer?]
Plan: Divide by 13 to get the per-region average
#E3 = Calculator[#E2 / 13]
"""

parsed = parse_plan(sample_plan)
print(f"Parsed {len(parsed)} steps:")
for s in parsed:
    # Show that #E references inside inputs are left as-is after parsing
    print(f"  {s.variable} = {s.tool}[{s.input}]")

Parsed 3 steps:
  #E1 = WikiSearch[France population]
  #E2 = LLMQuery[From this text: #E1 — what is France's population as a plain integer?]
  #E3 = Calculator[#E2 / 13]


## Generate a plan with a single LLM call
The planner is the first and most important phase of ReWOO. Given the query and the list of available tools, the LLM must identify every piece of information it will need and write down each lookup or calculation as a labelled step - all of this before any tool runs. Each step assigns its result to a numbered variable (`#E1`, `#E2`, ...) and may reference earlier variables inside its input string to express data dependencies.

The tool descriptions deliberately use the same `ToolName[input]` format as the plan steps. When the LLM reads the descriptions and the format specification side by side, it is far less likely to invent alternative syntax. The instruction "plan ALL steps upfront — no execution happens during planning" explicitly prevents the model from writing narrative prose or attempting to answer the question directly.

In [7]:
# Tool descriptions shown to the planner LLM — names must exactly match TOOLS dict keys
TOOLS_DESCRIPTION = (
    "WikiSearch[query] — search Wikipedia for factual information about a topic\n"
    "Calculator[expression] — evaluate a maths expression, e.g. Calculator[67_000_000 / 13]\n"
    "LLMQuery[question] — ask the LLM for reasoning, extraction, or analysis of earlier results"
)


def plan(query: str) -> List[PlanStep]:
    """
    Phase 1 of ReWOO: produce a complete execution plan with one LLM call.

    The LLM receives the query and tool descriptions and must output ALL required
    tool calls before any execution begins. Later steps may reference earlier
    results using #E variables inside their input strings.

    Args:
        query: The question to answer.

    Returns:
        Ordered list of PlanStep namedtuples representing the full plan.
    """
    system_msg = SystemMessage(content=(
        "You are a planning agent. Given a query, produce a complete step-by-step plan.\n\n"
        f"Available tools:\n{TOOLS_DESCRIPTION}\n\n"
        "Format each step as:\n"
        "Plan: <one sentence describing what this step does>\n"
        "#E<n> = ToolName[tool input]\n\n"
        "Rules:\n"
        "- Number variables #E1, #E2, #E3, ... in order\n"
        "- Place #E references inside [...] to chain results, "
        "e.g. LLMQuery[From #E1, extract the population as a number]\n"
        "- Do NOT add any text outside this format\n"
        "- Plan ALL steps upfront — no execution happens during planning"
    ))
    user_msg = HumanMessage(content=f"Query: {query}")

    response = llm.invoke([system_msg, user_msg])
    raw_plan = response.content

    print("LLM plan output:")
    print(raw_plan)

    return parse_plan(raw_plan)

- **`TOOLS_DESCRIPTION`** uses `ToolName[input]` format consistently with the plan syntax the LLM is asked to produce. This alignment reduces the chance the model mixes up brackets, parentheses, or key-value styles.
- **`plan()`** passes only two messages to the LLM: a system message defining the format and a user message with the query. The raw response is printed before parsing so any malformed output is immediately visible - if the parsed step count is zero, the raw output above explains why. The function is a thin wrapper around an LLM call followed by `parse_plan`; it carries no state of its own.

In [8]:
# Test the planner on a simple two-step question
test_query = "What is the area of Germany in square kilometres?"
test_steps = plan(test_query)

print(f"\nParsed {len(test_steps)} step(s):")
for s in test_steps:
    print(f"  {s.variable} = {s.tool}[{s.input}]")

LLM plan output:
Plan: Search for the area of Germany on Wikipedia to obtain accurate information.
#E1 = WikiSearch[area of Germany in square kilometres] 

Plan: Extract the area of Germany as a number from the search results.
#E2 = LLMQuery[From #E1, extract the area of Germany as a number]

Parsed 2 step(s):
  #E1 = WikiSearch[area of Germany in square kilometres]
  #E2 = LLMQuery[From #E1, extract the area of Germany as a number]


## Resolve variable references before calling each tool
Between planning and execution there is one critical operation: substituting the `#E` placeholders in each step's input string with the actual values computed by earlier steps. This is what makes the chain work - step 3 can reference the result of step 1 even though the plan was written before step 1 ran.

The substitution is a plain string replace across the whole input. If step 2's input is `"LLMQuery[From this text: #E1 — extract the number]"` and `#E1` resolved to a 200-word Wikipedia paragraph, substitution produces a fully concrete string ready to pass to the tool. This correctly handles references embedded inside a longer string - a case the original implementation's `param_value.startswith('#E')` check missed entirely.

One subtlety: substituting shorter variable names first risks partial matches. Replacing `#E1` before `#E10` would corrupt the prefix of `#E10`. Sorting variable names in reverse order ensures `#E10` is replaced before `#E1`, preventing this.

In [9]:
def substitute_variables(text: str, results: Dict[str, str]) -> str:
    """
    Replace every #En placeholder in text with its computed value from results.

    Substitutions are applied in reverse-sorted order so that longer variable
    names (e.g. #E10) are processed before their prefixes (e.g. #E1), preventing
    partial matches that would corrupt the string.

    Args:
        text: The raw input string for a plan step (may contain #E references).
        results: Map of variable name → computed string value from previous steps.

    Returns:
        The input string with all known #E references replaced by their values.
    """
    # Reverse sort ensures #E10 is replaced before #E1 (prevents prefix collision)
    for var in sorted(results.keys(), reverse=True):
        text = text.replace(var, str(results[var]))
    return text

**`substitute_variables`** is a single-responsibility function with no LLM calls - pure string manipulation. The reverse sort on variable names handles the edge case where names share a prefix (`#E1` and `#E10`). In practice with plans of three to five steps this edge case is unlikely, but the sort costs nothing and eliminates a subtle correctness bug.

The function does not validate whether all references were resolved. If a plan references `#E3` but step 3 failed, the literal string `#E3` remains in the next step's input. The tool call that follows may fail visibly - which is the desired behaviour, since it surfaces the upstream failure rather than silently producing wrong results.

## Execute the plan step by step
The execution phase is a simple loop: for each plan step, resolve variable references in the input string, look up the tool function, call it, and store the result under the step's variable name. No LLM is involved - this phase is pure function calls. The variable store (`results` dict) grows as each step completes, making each new result available for substitution in all subsequent steps.

The executor logs each step showing the *original* (unresolved) input from the plan - not the resolved one. A resolved input may be a 300-word Wikipedia paragraph embedded inside a question string; showing that would obscure the trace. The original input directly matches the plan line the LLM wrote, making it easy to compare the execution trace against the plan.

Errors in one step do not abort the loop. Recording the error message as the variable value means later steps that reference it will embed the error string in their inputs - which typically causes the next tool call to fail visibly and informatively, rather than crashing silently.

In [10]:
def execute(steps: List[PlanStep]) -> Dict[str, str]:
    """
    Phase 2 of ReWOO: run each plan step in order.

    Variable references in each step's input string are resolved with actual values
    from previous steps before the tool is called. This is how information flows
    between steps: #E1 in a later step's input becomes the full result of step 1.

    Args:
        steps: Ordered list of PlanStep objects from plan().

    Returns:
        Dict mapping variable names (#E1, #E2, ...) to their computed string results.
    """
    results: Dict[str, str] = {}

    for step in steps:
        # Replace all #E references with their actual computed values
        resolved_input = substitute_variables(step.input, results)

        # Look up the callable in the registry by the name the plan used
        tool_fn = TOOLS.get(step.tool)
        if tool_fn is None:
            # Unknown tool: store an error string and continue — later steps may still run
            results[step.variable] = f"[Error: unknown tool '{step.tool}']"
            print(f"  {step.variable}: unknown tool '{step.tool}' — skipping")
            continue

        try:
            result = str(tool_fn(resolved_input))
        except Exception as e:
            result = f"[Error: {e}]"

        results[step.variable] = result

        # Log with the ORIGINAL (unresolved) input so the trace matches the plan
        preview = result[:80] + "..." if len(result) > 80 else result
        print(f"  {step.variable} = {step.tool}[{step.input}]")
        print(f"    → {preview}")

    return results

**`execute()`** calls `substitute_variables` at the start of each loop iteration - not once upfront for the whole plan. This is required because `results` grows as each step completes: step 3's substitution can only resolve `#E2` after step 2 has run and populated that key.

The single most important correctness difference from the original implementation: substitution applies to the full input string, not just to parameter values whose entire content is a `#E` reference. The step `LLMQuery[From this text: #E1 — what is the population?]` correctly produces `LLMQuery[From this text: France is a country with population 67.75 million... — what is the population?]` after substitution. The original `param_value.startswith('#E')` check would have left `#E1` unresolved here, passing the literal placeholder to the LLM.

## Synthesise the final answer
After all tool calls complete, the `results` dict holds every piece of information the plan gathered. The third and final LLM call takes this collected evidence and the original query, and produces a coherent answer. This call is pure synthesis: the model is not asked to plan, execute, or retrieve - only to combine the given information into a direct response.

The context passed to the solver pairs each variable with its source (the tool name and original input string) and its computed result. Showing the source alongside the value helps the model understand where each fact came from, which improves grounding. The instruction "do not guess - only use the information provided" makes the boundary between retrieved facts and parametric knowledge explicit.

In [11]:
def solve(query: str, steps: List[PlanStep], results: Dict[str, str]) -> str:
    """
    Phase 3 of ReWOO: synthesise a final answer from the gathered information.

    This is the second and last LLM call in the pipeline. The model sees all
    collected results and is asked only to combine them into a clear answer.

    Args:
        query: The original question.
        steps: Plan steps — used to label each variable with its tool source.
        results: Computed values from execute().

    Returns:
        The final answer as a plain string.
    """
    # Build readable context: pair each variable's source with its computed value
    context = "\n\n".join(
        f"{step.variable} = {step.tool}[{step.input}]\n"
        f"Result: {results.get(step.variable, '(not computed)')}"
        for step in steps
    )

    system_msg = SystemMessage(content=(
        "You are answering a question using pre-gathered information. "
        "Use only the results provided — do not guess or add unsupported information. "
        "Be direct and complete."
    ))
    user_msg = HumanMessage(content=(
        f"Question: {query}\n\n"
        f"Gathered information:\n{context}\n\n"
        f"Answer:"
    ))

    response = llm.invoke([system_msg, user_msg])
    return response.content

**`solve()`** builds context by iterating over `steps` rather than `results`, so the variable order matches the plan order and each entry is paired with its tool source. This makes it easy for the LLM to understand the provenance of each fact - for example, `#E2 = LLMQuery[From #E1, extract the population]` followed by its result makes explicit that this number was extracted from Wikipedia, not generated from memory.

The system message explicitly prohibits guessing. Without this constraint, the LLM may blend plan results with its own parametric knowledge, which produces answers that are harder to trace and verify against the plan output.

## The ReWOO driver: plan, execute, solve
The three phases connect into a single function. The driver's only job is to call `plan()`, pass its output to `execute()`, and pass both outputs to `solve()` - plus print clear phase headers so the three stages are visible in the notebook output. There is no class, no state machine, no loop. Each phase communicates through plain return values: `plan()` returns a list, `execute()` returns a dict, `solve()` returns a string.

ReWOO's efficiency claim is visible at the code level: exactly two LLM invocations occur regardless of how many tools the plan contains - one in `plan()` and one in `solve()`. Every tool call in `execute()` is a direct Python function call with no LLM involved.

In [12]:
def rewoo(query: str) -> str:
    """
    Execute the full ReWOO pipeline: plan → execute → solve.

    Uses exactly two LLM calls regardless of how many tools the plan requires:
      - plan()    calls the LLM once to identify all tool calls upfront
      - execute() runs the tools with variable substitution (no LLM involved)
      - solve()   calls the LLM once to synthesise the gathered results

    Args:
        query: The question to answer.

    Returns:
        The final answer as a string.
    """
    print(f"Query: {query}")
    print("=" * 65)

    # ── Phase 1: plan ─────────────────────────────────────────────────────────
    print("\n[Phase 1 — Plan]")
    steps = plan(query)

    if not steps:
        # Parsing yielded zero steps — the LLM used an unrecognised format
        return "Planning failed: no steps were parsed. Check the LLM output above."

    print(f"\n{len(steps)} step(s) in the plan:")
    for s in steps:
        # Show the clean parsed plan (Plan: annotations stripped)
        print(f"  {s.variable} = {s.tool}[{s.input}]")

    # ── Phase 2: execute ───────────────────────────────────────────────────────
    print("\n[Phase 2 — Execute]")
    results = execute(steps)

    # ── Phase 3: solve ─────────────────────────────────────────────────────────
    print("\n[Phase 3 — Solve]")
    answer = solve(query, steps, results)

    print("\n" + "=" * 65)
    print("Answer:")
    print(answer)

    return answer

## Solving a multi-step problem with ReWOO
To see all three phases in action end to end, we run `rewoo()` on a query that requires a Wikipedia lookup, a number-extraction step, and an arithmetic calculation. This particular query is a good test case because the Calculator cannot operate directly on Wikipedia text - the plan must include an intermediate `LLMQuery` step to extract the numeric population before dividing it. If the planner writes `Calculator[#E1 / 13]` and `#E1` is a paragraph of prose text, the evaluation will fail. Observing whether the plan includes the extraction step is a useful check of plan quality.

In [13]:
query = (
    "What is the population of France, "
    "and what is that number divided by 13 (the number of metropolitan regions)?"
)

answer = rewoo(query)

Query: What is the population of France, and what is that number divided by 13 (the number of metropolitan regions)?

[Phase 1 — Plan]
LLM plan output:
Plan: First, I will search for the current population of France on Wikipedia.  
#E1 = WikiSearch[population of France]  

Plan: Next, I will extract the population number from the search results.  
#E2 = LLMQuery[From #E1, extract the population as a number]  

Plan: Then, I will divide the population number by 13 to find the average population per metropolitan region.  
#E3 = Calculator[#E2 / 13]  

3 step(s) in the plan:
  #E1 = WikiSearch[population of France]
  #E2 = LLMQuery[From #E1, extract the population as a number]
  #E3 = Calculator[#E2 / 13]

[Phase 2 — Execute]
  #E1 = WikiSearch[population of France]
    → The demography of France is monitored by the Institut national d'études démograp...
  #E2 = LLMQuery[From #E1, extract the population as a number]
    → The population of Metropolitan France as of 1 January 2026 is 66,79

The three phases are visible clearly in the output. The plan block shows the LLM's raw output followed by the clean parsed representation after `Plan:` annotations are stripped. The execute block shows one line per step in `#En = Tool[input] → result` format, where the input is the unresolved form from the plan. The solve block produces the final answer from those facts. Counting LLM calls across the entire run: exactly two, regardless of the number of steps.

ReWOO separates the two cognitive tasks an agent performs - deciding what to do, and interpreting what happened — into two distinct LLM calls. Everything in between (the tool execution) requires no LLM at all.

**When ReWOO fits well:**
- Queries where all required lookups can be identified before any tool runs.
- Multi-step research tasks where several independent facts must be gathered and combined.
- Situations where LLM calls are expensive and should be minimised.
- Pipelines where independent tool calls can be parallelised (steps whose inputs share no `#E` dependencies have no execution-time data dependency).

**When ReWOO is a poor fit:**
- Exploratory tasks where the next lookup depends on what the previous one found.
- Queries that require adaptive replanning - if step 2 reveals the question was misunderstood, a static plan cannot self-correct.
- Situations requiring conditional branching - e.g., "if the Wikipedia page exists, do X; otherwise do Y".

**The three functions and what each does:**
- `plan()` — one LLM call; produces the complete sequence of tool calls as `PlanStep` namedtuples.
- `execute()` — no LLM; runs each tool with `substitute_variables` resolving `#E` references from previous results.
- `solve()` — one LLM call; synthesises all variable values into a final answer.